In [16]:
# Cell 1 — Setup
import os
import glob
import shutil
import subprocess

from natsort import natsorted

# This notebook lives in notebooks/, but INPUT_DIR/OUTPUT_DIR are relative to the
# repo root. Walk up from the current working dir until we find the data folder,
# then chdir there so the notebook works regardless of where the kernel started.
def _find_repo_root(marker=os.path.join("data", "ru_cleaned")):
    d = os.getcwd()
    while True:
        if os.path.isdir(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError(
                f"Could not locate repo root containing {marker!r} starting from {os.getcwd()!r}"
            )
        d = parent


os.chdir(_find_repo_root())
print("Working directory:", os.getcwd())

# INPUT_DIR holds one cleaned .md per sub-document (72.1.337-<N>.ru.md). We translate
# each document in a single pass and write 72.1.337-<N>.en.md into OUTPUT_DIR —
# flat, no sub-folders.
INPUT_DIR = "data/ru_cleaned/"
OUTPUT_DIR = "data/en/"
MODEL = "claude-opus-4-8"   # switch to "claude-sonnet-4-6" for cheaper/faster
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fail fast if the Claude CLI is not on PATH.
CLAUDE_BIN = shutil.which("claude")
if CLAUDE_BIN is None:
    raise RuntimeError(
        "The `claude` CLI was not found on PATH. Install Claude Code and make sure "
        "`claude` is runnable from this kernel's environment."
    )
print("claude CLI:", CLAUDE_BIN)

Working directory: /Users/okolobaxa/Documents/Projects/neo4gen
claude CLI: /opt/homebrew/bin/claude


In [17]:
# Cell 2 — Discovery helper (natural sort via natsort)
# One cleaned file per sub-document: "72.1.337-<N>.ru.md". natsorted() orders by N
# numerically, so 72.1.337-2 precedes 72.1.337-13 (NOT lexicographic).


def sorted_docs(input_dir):
    return natsorted(glob.glob(os.path.join(input_dir, "*.ru.md")))


print("Documents to translate:", [os.path.basename(p) for p in sorted_docs(INPUT_DIR)])

Documents to translate: ['0-0-0.ru.md', '28.1.2540.ru.md', '72.1.337.ru.md']


In [18]:
# Cell 3 — Translation function (one `claude` CLI call per cleaned document)

SYSTEM_PROMPT = """You are an expert translator of 19th-century Russian archival
documents. You are given the CLEANED text of a single Russian
administrative/clerical document written in pre-reform orthography (it spans
several consecutive pages). Translate it into English.

Your task — produce a faithful, readable translation of the whole document:
- Translate into clear, plain modern English that accurately preserves the meaning.
- Preserve proper names; transliterate personal and place names consistently across
  the whole document (e.g. Харашевскій → Kharashevsky, Якубов → Yakubov).
- Keep document structure (paragraphs) intact; do not reorder content.
- If a passage has a marginal block under the line "> [на полях]", translate that
  block too and place it under the line "> [in margin]" at the end of the
  corresponding fragment.

STRICTLY OBSERVE:
- Do NOT invent, embellish, or add text. Leave genuinely illegible or uncertain
  passages as they are (you may keep them in the original or mark them [illegible]),
  but do not guess at meaning that is not there.
- Output ONLY the English translation of the document. No preamble, no commentary, no
  enclosing ``` fences."""


def _strip_fences(text):
    """Defensively remove a leading/trailing ``` code fence if the model wrapped output."""
    s = text.strip()
    if s.startswith("```"):
        lines = s.split("\n")
        lines = lines[1:]  # drop opening fence line (``` or ```lang)
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]  # drop closing fence line
        s = "\n".join(lines)
    return s


def translate_document(ru_text):
    proc = subprocess.run(
        [CLAUDE_BIN, "-p", f"DOCUMENT TO TRANSLATE:\n{ru_text}", "--model", MODEL,
         "--append-system-prompt", SYSTEM_PROMPT],
        capture_output=True, text=True, timeout=1800,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"claude failed (rc={proc.returncode}): {proc.stderr.strip()}")
    return _strip_fences(proc.stdout).strip()

In [19]:
# Cell 4 — Translation loop
# --- Config ---
LIMIT = None        # process only the first N documents; set None to process all
REPROCESS = False   # True = re-translate and overwrite even if an output file already exists

from IPython.display import display

files = sorted_docs(INPUT_DIR)
if LIMIT is not None:
    files = files[:LIMIT]

total = len(files)
display(f"Translating {total} document(s)  (LIMIT={LIMIT}, REPROCESS={REPROCESS}, MODEL={MODEL})")

for i, input_path in enumerate(files, start=1):
    name = os.path.basename(input_path)
    out_name = name.replace(".ru.md", ".en.md")
    out_path = os.path.join(OUTPUT_DIR, out_name)

    if os.path.exists(out_path) and not REPROCESS:
        display(f"[{i}/{total}] skip  {out_name} (already translated)")
        continue

    display(f"[{i}/{total}] translate {name} ...")
    # No try/except by design: an error halts the loop so the failing document can be
    # inspected. Re-running later skips completed files and resumes here.
    with open(input_path, encoding="utf-8") as f:
        ru_text = f.read()
    translated = translate_document(ru_text)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(translated)
    display(f"[{i}/{total}] done  {out_name}")

display("Finished.")

'Translating 3 document(s)  (LIMIT=None, REPROCESS=False, MODEL=claude-opus-4-8)'

'[1/3] translate 0-0-0.ru.md ...'

'[1/3] done  0-0-0.en.md'

'[2/3] translate 28.1.2540.ru.md ...'

'[2/3] done  28.1.2540.en.md'

'[3/3] translate 72.1.337.ru.md ...'

'[3/3] done  72.1.337.en.md'

'Finished.'